# LSTM-QGAN — IBM Quantum Hardware Validation

Loads trained generator weights, extracts VQC parameters from one QLSTM gate,
runs the same 7-qubit circuit on PennyLane simulator and a real IBM backend,
then compares PauliZ expectation values between the two.

> Proof-of-concept circuit validation — not image generation on hardware.

In [ ]:
# uncomment if packages are missing
!pip install numpy torch matplotlib pennylane qiskit "qiskit-ibm-runtime>=0.20" scipy

In [1]:
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pennylane as qml
from qiskit import QuantumCircuit
from qiskit.circuit.library import StatePreparation
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

print('imports OK')

imports OK


In [ ]:
CHECKPOINT_PATH = r'local_gpu\outputs_long\generator_final.pth'
IBM_API_KEY = 'PASTE_YOUR_IBM_API_KEY_HERE'

N_QUBITS  = 7
N_LAYERS  = 2
N_SAMPLES = 16
N_SHOTS   = 1024
SEED      = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
state = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
if isinstance(state, dict) and 'generator' in state:
    state = state['generator']

REF_KEY   = 'qlstm1.cells.0.qnn_forget.weights'
W_trained = state[REF_KEY].detach().numpy()
print(f'weights shape: {W_trained.shape}  range: [{W_trained.min():.3f}, {W_trained.max():.3f}]')

weights shape: (2, 7, 3)  range: [0.206, 5.996]


In [4]:
_HILBERT = 2 ** N_QUBITS
dev_sim  = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev_sim, interface='numpy')
def pennylane_circuit(inputs, weights):
    qml.AmplitudeEmbedding(features=inputs, wires=range(N_QUBITS), normalize=True, pad_with=0.0)
    for layer in range(N_LAYERS):
        for q in range(N_QUBITS):
            qml.RX(weights[layer, q, 0], wires=q)
            qml.RY(weights[layer, q, 1], wires=q)
            qml.RZ(weights[layer, q, 2], wires=q)
        for q in range(N_QUBITS):
            qml.CNOT(wires=[q, (q + 1) % N_QUBITS])
    return [qml.expval(qml.PauliZ(q)) for q in range(N_QUBITS)]

rng = np.random.default_rng(SEED)
raw = rng.standard_normal((N_SAMPLES, _HILBERT)).astype(np.float64)
inputs_norm = raw / np.linalg.norm(raw, axis=-1, keepdims=True).clip(min=1e-8)

sim_results = np.array([pennylane_circuit(inputs_norm[i], W_trained) for i in range(N_SAMPLES)])
print(f'sim shape: {sim_results.shape}')
print(f'mean per qubit: {sim_results.mean(0).round(4)}')

sim shape: (16, 7)
mean per qubit: [ 0.032  -0.0371 -0.0056 -0.0074 -0.0039 -0.0278 -0.0138]


In [5]:
def build_bound_circuit(amp_vec, weights):
    qc  = QuantumCircuit(N_QUBITS, N_QUBITS)
    vec = amp_vec.astype(float)
    vec = vec / np.linalg.norm(vec).clip(min=1e-8)
    qc.append(StatePreparation(vec.tolist(), label='AmpEmbed'), range(N_QUBITS))
    for layer in range(N_LAYERS):
        for q in range(N_QUBITS):
            qc.rx(float(weights[layer, q, 0]), q)
            qc.ry(float(weights[layer, q, 1]), q)
            qc.rz(float(weights[layer, q, 2]), q)
        for q in range(N_QUBITS):
            qc.cx(q, (q + 1) % N_QUBITS)
    qc.measure_all(add_bits=False)
    return qc

circuits = [build_bound_circuit(inputs_norm[i], W_trained) for i in range(N_SAMPLES)]
print(f'built {len(circuits)} circuits  depth: {circuits[0].depth()}')

built 16 circuits  depth: 22


In [6]:
# instance is not needed — the SDK picks your default from the saved account
QiskitRuntimeService.save_account(
    channel='ibm_quantum_platform',
    token=IBM_API_KEY,
    overwrite=True,
)
service = QiskitRuntimeService(channel='ibm_quantum_platform')
backend = service.least_busy(min_num_qubits=N_QUBITS, simulator=False)
print(f'backend: {backend.name}  qubits: {backend.num_qubits}')

qiskit_runtime_service.__init__:WARNING:2026-04-25 13:05:46,919: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-25 13:05:47,721: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-04-25 13:05:48,968: Using instance: open-instance, plan: open


backend: ibm_fez  qubits: 156


In [7]:
pm         = generate_preset_pass_manager(backend=backend, optimization_level=1)
circuits_t = pm.run(circuits)
print(f'transpiled {len(circuits_t)} circuits  depth: {circuits_t[0].depth()}')

transpiled 16 circuits  depth: 987


In [8]:
sampler = Sampler(mode=backend)
job     = sampler.run([(c,) for c in circuits_t], shots=N_SHOTS)
print(f'job ID: {job.job_id()}')
print('waiting for results...')

job ID: d7m6tmraq2pc73a1bavg
waiting for results...


In [9]:
def counts_to_pauliz(counts, n_qubits=N_QUBITS):
    expvals = np.zeros(n_qubits)
    total   = sum(counts.values())
    for bits, count in counts.items():
        bits = bits.replace(' ', '')[-n_qubits:]
        for qi, b in enumerate(reversed(bits)):
            expvals[qi] += (1 - 2 * int(b)) * count
    return expvals / total

result = job.result()

real_results = []
for pr in result:
    # grab whichever classical register the transpiler used
    reg_name = list(pr.data.keys())[0]
    counts   = pr.data[reg_name].get_counts()
    real_results.append(counts_to_pauliz(counts))

real_results = np.array(real_results)
print(f'real shape: {real_results.shape}')
print(f'mean (real): {real_results.mean(0).round(4)}')
print(f'mean (sim) : {sim_results.mean(0).round(4)}')

np.save('sim_results.npy',  sim_results)
np.save('real_results.npy', real_results)

real shape: (16, 7)
mean (real): [ 0.015   0.0176  0.036   0.0157  0.0083  0.0222 -0.0168]
mean (sim) : [ 0.032  -0.0371 -0.0056 -0.0074 -0.0039 -0.0278 -0.0138]


In [10]:
mae    = np.abs(sim_results - real_results).mean(axis=0)
errors = real_results - sim_results

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.scatter(sim_results.ravel(), real_results.ravel(), alpha=0.5, s=20)
ax.plot([-1.1, 1.1], [-1.1, 1.1], 'r--', lw=1)
ax.set(xlim=[-1.1,1.1], ylim=[-1.1,1.1], xlabel='Simulator', ylabel='Real hardware', title='PauliZ comparison')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(range(N_QUBITS), mae, color='coral', edgecolor='black')
ax.axhline(mae.mean(), color='navy', linestyle='--', label=f'mean={mae.mean():.3f}')
ax.set(xlabel='Qubit', ylabel='MAE', title='Per-qubit MAE')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

ax = axes[2]
im = ax.imshow(errors, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set(xlabel='Qubit', ylabel='Sample', title='Signed error (real - sim)')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('LSTM-QGAN VQC: Simulator vs IBM Quantum Hardware', fontweight='bold')
plt.tight_layout()
plt.savefig('real_hw_noise_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'MAE: {mae.mean():.4f}')

MAE: 0.0871


C:\Users\pthar\AppData\Local\Temp\ipykernel_39776\3145745202.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results

| Panel | What it shows |
|-------|---------------|
| Scatter | Sim vs real PauliZ — points near y=x = low noise impact |
| MAE bar | Per-qubit mean absolute error (typical: 0.05–0.25 on free backends) |
| Heatmap | Signed error per sample/qubit — shows where noise hits hardest |

A successful run confirms the trained QLSTM circuit is hardware-compatible.